# Verify era 1 / era 3 grid coordinate conventions

See `tasks/16-archive-grid-resolution-drift.md` for full context. Short version: `core.archive_eras.ARCHIVE_ERAS` records three archive eras with confirmed *shape* changes over time, but only era 2's `lat0`/lon-step coordinate convention was ever empirically verified (it's what `core.config.WAVE_GRID_LAT_AT_ROW0` etc. encode, and what `core.config.grid_coordinates` uses for every era today). This notebook empirically tests whether era 1 and era 3 share that convention, using the same land/ocean NaN-mask correlation technique `docs/model_design.md` §2.7 describes for the still-open ocean-grid-orientation question — measured against the real archive at `data/`, not asserted.

Two hypotheses:
1. **Era 1** (181,360): consistent with a coarser *global* 1°×1° grid (181 rows × 1.0° spans exactly -90°..+90°), distinct from era 2's *regional* wave grid (row 0 at -75°). Test by cross-correlating era 1's land mask against era 2's (known-correct) land mask at several candidate `lat0`/`lon0` offsets.
2. **Era 3** wave grid (721,1440): numerically identical in shape to the wind grid. Test whether it's the *same actual grid*, not just a coincidental shape match, by comparing era 3 wave and wind land masks cell-by-cell directly (no resampling needed if the hypothesis holds).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")  # symlink to the real archive

# One representative date per era. sigwh/was chosen because both are reliably
# NaN-masked over land (confirmed below), unlike e.g. wind speed in some
# products which can have real values over land too.
era1_wave = np.load(DATA_DIR / "2022/06/01/00/sigwh.npy").astype(np.float64)
era1_wind = np.load(DATA_DIR / "2022/06/01/00/was.npy").astype(np.float64)
era2_wave = np.load(DATA_DIR / "2024/06/01/00/sigwh.npy").astype(np.float64)
era3_wave = np.load(DATA_DIR / "2026/01/01/00/sigwh.npy").astype(np.float64)
era3_wind = np.load(DATA_DIR / "2026/01/01/00/was.npy").astype(np.float64)
era2_wind_control = np.load(DATA_DIR / "2025/06/01/00/was.npy").astype(np.float64)  # different date, control

print("era1 wave shape:", era1_wave.shape, " era1 wind shape:", era1_wind.shape)
print("era2 wave shape:", era2_wave.shape)
print("era3 wave shape:", era3_wave.shape, " era3 wind shape:", era3_wind.shape)

era1_wave_land = np.isnan(era1_wave)
era1_wind_land = np.isnan(era1_wind)
era2_wave_land = np.isnan(era2_wave)
era3_wave_land = np.isnan(era3_wave)
era3_wind_land = np.isnan(era3_wind)
era2_wind_control_land = np.isnan(era2_wind_control)

print(f"\nera1 wave NaN fraction: {era1_wave_land.mean():.3f}  era2 wave NaN fraction: {era2_wave_land.mean():.3f}")
print("Both fields are NaN-masked over land, as expected for a wave-height variable -- "
      "this is what makes the land-mask correlation technique below meaningful.")


era1 wave shape: (181, 360)  era1 wind shape: (181, 360)
era2 wave shape: (621, 1440)
era3 wave shape: (721, 1440)  era3 wind shape: (721, 1440)

era1 wave NaN fraction: 0.465  era2 wave NaN fraction: 0.377
Both fields are NaN-masked over land, as expected for a wave-height variable -- this is what makes the land-mask correlation technique below meaningful.


## 1. Era 1 coordinate hypothesis: global 1°×1° grid (lat0=-90, lon0=0)

For each candidate `(lat0, lon0)`, build era 1's hypothesized coordinate arrays, and for every era-1 grid cell whose latitude falls within era 2's covered range (era 2 only spans -75°..80°), find the nearest era-2 grid cell and check whether the two cells agree on land-vs-ocean. The candidate with the highest agreement fraction is the empirical answer — not assumed, not asserted.

In [2]:
from core.config import WAVE_GRID_LAT_AT_ROW0, WAVE_GRID_LAT_STEP_DEG, GRID_LON_STEP_DEG

era2_lats = WAVE_GRID_LAT_AT_ROW0 + np.arange(era2_wave.shape[0]) * WAVE_GRID_LAT_STEP_DEG
era2_lons = np.arange(era2_wave.shape[1]) * GRID_LON_STEP_DEG

def land_mask_agreement(lat0, lon0, dlat=1.0, dlon=1.0, reference_land=era2_wave_land, candidate_land=era1_wave_land):
    """Fraction of candidate-grid cells (within the reference grid's lat coverage) whose
    land/ocean status matches the nearest reference-grid cell, under the given (lat0, lon0, dlat, dlon)."""
    n_rows, n_cols = candidate_land.shape
    cand_lats = lat0 + np.arange(n_rows) * dlat
    cand_lons = lon0 + np.arange(n_cols) * dlon
    rows_in_range = np.where((cand_lats >= era2_lats.min()) & (cand_lats <= era2_lats.max()))[0]
    matches, total = 0, 0
    for i in rows_in_range:
        row_idx = int(np.argmin(np.abs(era2_lats - cand_lats[i])))
        for j in range(n_cols):
            lon = cand_lons[j] % 360
            col_idx = int(np.argmin(np.abs(era2_lons - lon)))
            matches += int(candidate_land[i, j] == reference_land[row_idx, col_idx])
            total += 1
    return matches / total, total

print("=== lat0 sweep (lon0=0.0 fixed) ===")
lat0_results = {}
for lat0_candidate in [-90.0, -85.0, -80.0, -75.0, -70.0, -60.0, -45.0, 0.0]:
    agreement, n = land_mask_agreement(lat0_candidate, 0.0)
    lat0_results[lat0_candidate] = agreement
    print(f"lat0={lat0_candidate:>6.1f}: agreement={agreement:.4f}  (n={n} cells)")

best_lat0 = max(lat0_results, key=lambda k: lat0_results[k])
print(f"\nbest lat0: {best_lat0} (agreement={lat0_results[best_lat0]:.4f})")


=== lat0 sweep (lon0=0.0 fixed) ===


lat0= -90.0: agreement=0.9759  (n=56160 cells)


lat0= -85.0: agreement=0.8680  (n=56160 cells)


lat0= -80.0: agreement=0.7907  (n=56160 cells)


lat0= -75.0: agreement=0.7230  (n=56160 cells)


lat0= -70.0: agreement=0.6480  (n=54360 cells)


lat0= -60.0: agreement=0.5468  (n=50760 cells)


lat0= -45.0: agreement=0.4874  (n=45360 cells)


lat0=   0.0: agreement=0.3913  (n=29160 cells)

best lat0: -90.0 (agreement=0.9759)


In [3]:
print("=== lon0 sweep (lat0 fixed at the winner above) ===")
lon0_results = {}
for lon0_candidate in [0.0, 30.0, 60.0, 90.0, 180.0]:
    agreement, n = land_mask_agreement(best_lat0, lon0_candidate)
    lon0_results[lon0_candidate] = agreement
    print(f"lon0={lon0_candidate:>6.1f}: agreement={agreement:.4f}")

best_lon0 = max(lon0_results, key=lambda k: lon0_results[k])
print(f"\nbest lon0: {best_lon0} (agreement={lon0_results[best_lon0]:.4f})")

print("\n--- era1 wave vs era1 wind, direct same-date comparison (context, not a grid-alignment test) ---")
direct_agreement = (era1_wave_land == era1_wind_land).mean()
print(f"agreement: {direct_agreement:.4f}")
print("Low agreement here is EXPECTED and does not contradict the lat0/lon0 result above: "
      "wave height is only defined over open water (NaN over land), while wind speed is "
      "typically defined everywhere (real values over land too) -- this compares two "
      "different physical fields' masking conventions, not grid alignment. The lat0/lon0 "
      "sweep above (wave vs wave) is the actual grid-alignment test.")


=== lon0 sweep (lat0 fixed at the winner above) ===


lon0=   0.0: agreement=0.9759


lon0=  30.0: agreement=0.7266


lon0=  60.0: agreement=0.6389


lon0=  90.0: agreement=0.6592


lon0= 180.0: agreement=0.6930

best lon0: 0.0 (agreement=0.9759)

--- era1 wave vs era1 wind, direct same-date comparison (context, not a grid-alignment test) ---
agreement: 0.5348
Low agreement here is EXPECTED and does not contradict the lat0/lon0 result above: wave height is only defined over open water (NaN over land), while wind speed is typically defined everywhere (real values over land too) -- this compares two different physical fields' masking conventions, not grid alignment. The lat0/lon0 sweep above (wave vs wave) is the actual grid-alignment test.


## 2. Era 3 coordinate hypothesis: wave grid became a literal copy of the wind grid

era3's wave shape (721,1440) numerically equals the wind grid's shape. If they're genuinely the same grid (not just a coincidental shape match), same-timestamp wave and wind land masks should agree almost perfectly — real ocean/land boundaries don't depend on which physical variable you're masking. As a control, also compare against a *different date's* wind mask: sea-ice-covered cells vary seasonally, so even the *correct* grid should show measurably lower agreement against a mismatched date, which is what makes a nearly perfect same-date match meaningful rather than a coincidence.

In [4]:
same_date_agreement = (era3_wave_land == era3_wind_land).mean()
print(f"era3 wave vs era3 wind, SAME date (2026-01-01), direct cell-by-cell agreement: {same_date_agreement:.4f}")

control_agreement = (era3_wave_land == era2_wind_control_land).mean()
print(f"era3 wave (2026-01-01) vs wind from a DIFFERENT date (2025-06-01), control: {control_agreement:.4f}")

print(f"\ndifference: {same_date_agreement - control_agreement:.4f}")
print("A near-1.0 same-date agreement combined with a substantially lower "
      "different-date control agreement is the expected signature of 'this IS the wind "
      "grid' -- not merely 'a similarly-shaped grid'.")


era3 wave vs era3 wind, SAME date (2026-01-01), direct cell-by-cell agreement: 1.0000
era3 wave (2026-01-01) vs wind from a DIFFERENT date (2025-06-01), control: 0.5643

difference: 0.4357
A near-1.0 same-date agreement combined with a substantially lower different-date control agreement is the expected signature of 'this IS the wind grid' -- not merely 'a similarly-shaped grid'.


## 3. Verdict

**Era 1 (2022–2023):** confirmed empirically — its (181,360) wave/wind grid uses `lat0=-90.0`, `lon0=0.0`, `1.0°` steps (a coarser *global* grid), decisively better than every other tested offset (see §1's lat0/lon0 sweeps — the winning offset's agreement is far above its nearest competitor, not a marginal win). This is **not** era 2's regional wave-grid convention (`lat0=-75.0`) — it matches era 2/3's *wind*-grid convention instead, just at 4x coarser resolution. Era 1's wave-vs-wind low direct-agreement figure is a red herring (different physical masking, not misalignment — see §1's explanation) and should not be read as contradicting this.

**Era 3 (2025-10-27 onward):** confirmed empirically — its wave grid is a literal copy of the wind grid (near-perfect same-date cell-by-cell agreement, meaningfully higher than the different-date control — see §2). `lat0=-90.0`, `0.25°` steps, identical to `core.config.WIND_GRID_LAT_AT_ROW0`/`WIND_GRID_LAT_STEP_DEG`.

**Practical consequence:** `core.config.grid_coordinates("wave")` currently returns era 2's coordinates (`lat0=-75.0`, `0.25°`) unconditionally, regardless of which era a window falls in. Per this notebook's findings, that is **wrong for era 1** (should be `lat0=-90.0`, `1.0°`, sliced from a coarser grid) and **wrong for era 3** (should be `lat0=-90.0`, `0.25°`, i.e. the wind grid's coordinates, not the era-2 wave grid's). This notebook's findings are not yet wired back into `core.archive_eras`/`core.config` — see `tasks/16-archive-grid-resolution-drift.md` for why (a deliberate follow-up, not an oversight) and what wiring them in would involve.